In [ ]:
# ================================
# D2B model surface + overlay of actual points
# Yu-Ting Kao (Nov 2025)
# ================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import Normalize
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from adjustText import adjust_text
from mpl_toolkits.axes_grid1 import make_axes_locatable  # optional for 2nd colorbar

# ----------------------------
# User parameters
# ----------------------------
# Model (background surface)
ASSAY_CONC_uM = 10.0            # test dose (e.g., 5 µM)
SM_IC50_uM    = 11.5            # starting material IC50 (µM)
RESPONSE_MODEL = "emax"        # "emax" or "linear"
COMBINATION_MODE = "loewe"     # "loewe" (same axis), "bliss" (independent), "additive"
DEGREE = 2                     # polynomial degree for surface smoothing (2 is usually enough)

# Surface axis limits (in nM on a log x-axis)
XMIN_NM = 10**3                # e.g., 10^1 nM
XMAX_NM = 10**5                # e.g., 10^5 nM
YMIN_PCT = None                # None -> auto from data
YMAX_PCT = None

# File paths
BASE_CSV = 'RS_SMactive_v8_bioactive_SMic50 dependent-27.csv'            # background grid data
OVERLAY_CSV = 'Scatter plot list_20251030-all purified_N1only.csv'       # points to overlay

# Overlay config
OVERLAY_SIZE = 60
OVERLAY_EDGE = 'grey'
OVERLAY_EDGE_LW = 0.7
LABEL_COL = 'number'           # which column to use for labels (fallback to blank if missing)
USE_SECOND_COLORBAR = False    # set True to show a separate colorbar for the overlay points

# ----------------------------
# Load data for model surface
# (expects columns: 'ic50' in µM for PRODUCT and 'yield' in %)
# ----------------------------
df = pd.read_csv(BASE_CSV)

# Units
df['ic50_nM'] = df['ic50'] * 1000.0
SM_IC50_nM = SM_IC50_uM * 1000.0
ASSAY_CONC_nM = ASSAY_CONC_uM * 1000.0

# In-well concentrations at chosen dose
df['prod_conc_nM'] = ASSAY_CONC_nM * (df['yield'] / 100.0)
df['sm_conc_nM']   = ASSAY_CONC_nM * (1.0 - df['yield'] / 100.0)

# ----------------------------
# Single-agent models
# ----------------------------
def contrib_linear(C_nM, IC50_nM):
    return 100.0 * (C_nM / IC50_nM)

def contrib_emax(C_nM, IC50_nM):
    return 100.0 * (C_nM / (C_nM + IC50_nM))

if RESPONSE_MODEL.lower() == "emax":
    prod_eff = contrib_emax(df['prod_conc_nM'], df['ic50_nM'])
    sm_eff   = contrib_emax(df['sm_conc_nM'],   SM_IC50_nM)
else:
    prod_eff = contrib_linear(df['prod_conc_nM'], df['ic50_nM'])
    sm_eff   = contrib_linear(df['sm_conc_nM'],   SM_IC50_nM)

# Clip single-agent effects
prod_eff = np.clip(prod_eff, 0, 100)
sm_eff   = np.clip(sm_eff,   0, 100)

# ----------------------------
# Combination models → total_d2b_activity
# ----------------------------
mode = COMBINATION_MODE.lower()
if mode == "loewe":
    # Loewe additivity (Hill=1, Emax=100): S = sum(C/IC50); E = 100*S/(1+S)
    S = (df['prod_conc_nM'] / df['ic50_nM']) + (df['sm_conc_nM'] / SM_IC50_nM)
    total = 100.0 * (S / (1.0 + S))
elif mode == "bliss":
    # Bliss independence: Etot = A + B − A*B
    A = prod_eff / 100.0
    B = sm_eff   / 100.0
    total = 100.0 * (A + B - A * B)
else:
    # Simple additive (capped)
    total = prod_eff + sm_eff

df['total_d2b_activity'] = np.clip(total, 0, 100)

# ----------------------------
# Fit polynomial surface and predict on grid
# ----------------------------
x = df['ic50_nM'].values
y = df['yield'].values
z = df['total_d2b_activity'].values

X = np.vstack((np.log10(x), y)).T
model = make_pipeline(PolynomialFeatures(DEGREE), LinearRegression())
model.fit(X, z)

x_grid = np.linspace(np.log10(10**3), np.log10(10**5), 500)
y_lo = y.min() if YMIN_PCT is None else YMIN_PCT
y_hi = y.max() if YMAX_PCT is None else YMAX_PCT
y_grid = np.linspace(y_lo, y_hi, 500)

x_mesh_log, y_mesh = np.meshgrid(x_grid, y_grid)
grid_points = np.vstack((x_mesh_log.ravel(), y_mesh.ravel())).T

z_pred = model.predict(grid_points).reshape(x_mesh_log.shape)
z_pred = np.clip(z_pred, 0, 100)

# Colormap & norms
cmap_surface = 'plasma'
norm_surface = Normalize(vmin=np.nanmin(z_pred), vmax=100)

# ----------------------------
# Plot background surface
# ----------------------------
fig, ax = plt.subplots(figsize=(8, 6))
pcm = ax.pcolormesh(10**x_mesh_log, y_mesh, z_pred, cmap=cmap_surface, shading='auto', norm=norm_surface)
cbar = fig.colorbar(pcm, ax=ax, label='D2B Activity (%)')

def scientific_notation(xval, pos):
    return f'$10^{{{int(np.log10(xval))}}}$' if xval != 0 else '0'
ax.xaxis.set_major_formatter(FuncFormatter(scientific_notation))
ax.set_xscale('log')
ax.set_xlim(XMIN_NM, XMAX_NM)
if YMIN_PCT is not None or YMAX_PCT is not None:
    ax.set_ylim(y_lo, y_hi)

plt.rc('font', size=10, family='Arial')
ax.set_xlabel('Product IC50 (nM)')
ax.set_ylabel('Yield (%)')
ax.set_title(f'D2B Total Activity @ {ASSAY_CONC_uM:g} µM '
             f'({COMBINATION_MODE.upper()} combo; SM IC50={SM_IC50_uM:g} µM)')

# ----------------------------
# OVERLAY: actual D2B points
# ----------------------------
df2 = pd.read_csv(OVERLAY_CSV)

# Harmonize IC50 for plotting (nM)
if 'ic50_nM' in df2.columns:
    df2['ic50_plot'] = df2['ic50_nM']
elif 'Ic50 in 72 hrs' in df2.columns:
    df2['ic50_plot'] = df2['Ic50 in 72 hrs'] * 1000.0  # µM → nM
else:
    raise ValueError("Overlay CSV must contain 'ic50_nM' or 'Ic50 in 72 hrs'.")

# Harmonize yield
if 'yield' not in df2.columns:
    if 'prod conversion %' in df2.columns:
        df2 = df2.rename(columns={'prod conversion %': 'yield'})
    else:
        raise ValueError("Overlay CSV must contain 'yield' or 'prod conversion %'.")

# Require D2B inhibition % for coloring
if 'D2B inhibition %' not in df2.columns:
    raise ValueError("Overlay CSV must contain 'D2B inhibition %' column for coloring.")

# Clean
df2 = df2.replace([np.inf, -np.inf], np.nan).dropna(subset=['ic50_plot', 'yield', 'D2B inhibition %'])
df2 = df2[(df2['ic50_plot'] > 0) & (df2['yield'].between(1, 100))]

# Overlay color norm (0–100)
norm_points = Normalize(vmin=0, vmax=100)
sc = ax.scatter(
    df2['ic50_plot'],
    df2['yield'],
    s=OVERLAY_SIZE,
    c=df2['D2B inhibition %'],
    cmap=cmap_surface,
    norm=norm_points,
    edgecolors=OVERLAY_EDGE,
    linewidths=OVERLAY_EDGE_LW,
    zorder=3
)

# Optional 2nd colorbar just for points
if USE_SECOND_COLORBAR:
    divider = make_axes_locatable(ax)
    cax2 = divider.append_axes("right", size="3%", pad=0.35)
    fig.colorbar(sc, cax=cax2, label='Overlay: D2B inhibition (%)')

# Labels with auto-adjust (uses LABEL_COL if present)
texts = []
label_series = df2[LABEL_COL] if LABEL_COL in df2.columns else pd.Series([''] * len(df2))
for (_, r), label in zip(df2.iterrows(), label_series):
    texts.append(ax.text(r['ic50_plot'], r['yield'], str(label), fontsize=10, ha='center', va='bottom', color='black'))

adjust_text(
    texts,
    x=df2['ic50_plot'],
    y=df2['yield'],
    arrowprops=dict(arrowstyle='-', color='gray', lw=1.0)
)

# Save / show
#plt.savefig(f'D2B_Surface_{ASSAY_CONC_uM:g}uM_{COMBINATION_MODE}_with_overlay.png', dpi=300, bbox_inches='tight')
plt.show()
